# 🐦 **Sử dụng LangChain để xây dụng quá trình Word Embedding**

## 📑 **Khái niệm**

- **LLM (Ngôn ngữ lớn)**

Là các mô hình AI được huấn luyện trên lượng dữ liệu văn bản cực lớn dể hiểu và tạo ra các ngôn ngữ tự nhiên

Ví dụ: *ChatGPT, GPT, Gemini*

Học bằng cách: Đọc rấy nhiều văn ban, học cách dự đoán từ tiếp theo trong câu

- **Vector Search**

Là cách tìm kiếm dựa trên ý nghĩa thay vì chỉ từ khóa

**Ý tưởng**

```
  - Mỗi đoạn văn / câu / từ được biến thành vector (một dãy số)
  - Những nội dung có ý nghĩa giống nhau -> vector sẽ gần nhau
```

**Vector trông như thế nào**

```
"king"  → [0.25, -0.11, 0.78, ...]
"queen" → [0.27, -0.10, 0.80, ...]
```

Hai vector gần nhau sẽ giống nhau

- **Word2Vec**
  - Là một trong những kỹ thuật nền tảng giúp máy tính hiểu ý nghĩa của từ bằng cách biến từ thành vector (Là mô hình do Google phát triển).
  - Ý tưởng là học từ ngữ cảnh của từ:
  ```
  Ví dụ: "Tôi nuôi một con mèo"
  
  Mô hình sẽ học:
   - "mèo" thường xuất hiện gần "nuôi", "con"
   - từ đó hiểu ý nghĩa của từ "mèo"
  ```
  - Hai cách huán luyện chính:
    - CBOW: Đoán từ ở giữa dựa vào từ xung quanh
    - Skip-gram: Từ trung tam đoán từ xung quanh

⇒ **Word2Vec sẽ biến các từ thành các vector sau đó sử dụng VectorSearch để tìm kiếm các từ cùng nghĩa**

---

## 🎰 **Kỹ thuật**

### ✂️ **Chia nhỏ văn bản hình thành vector database**

**Pipeline:**

```
Load tài liệu ->  Cắt nhỏ tài liệu -> Lưu và vector database
```

- **TextLoader**: Đọc file txt -> Biến text thành nội dung mà LangChain hiểu được

> Là bước lấy dữ liệu thô vào hệ thống AI

- **CharacterTextSplitter**: Chia nhỏ văn bản (cắt đoạn dài thành nhiều đoạn nhỏ)

- **Chroma**: Lưu các đoạn văn dưới dạng vector và cho phép tìm kiếm theo ngữ nghĩa

### 🧠 **Xây dựng Hệ thống**

**Pipeline**

```
Lây vector trong database -> Lưu và tìm kiếm vector tương đồng
```

- **HuggingFaceEmbeddings:** Biến text thành vector
- **Chroma:** Lưu và tìm kiếm vector -> so sánh vector và trả về đoạn giống nhất

### 🧩 **Flow hoàn chính trong notebook**

```
- Lớp biến đổi

File csv -> Chuyển thành file text -> Chia nhỏ văn bản -> Embedding - Chuyển văn bản thành vector (Hugging Face) -> Lưu vào vector database (Chroma)

- Lớp truy vấn

Query -> Embedding (vector) -> So sánh cosine similarity -> Top K kết quả

```




In [ ]:
!pip install langchain langchain_community langchain_chroma

# ✂️ **Chia nhỏ văn bản hình thành vector database**

In [ ]:
from langchain_community.document_loaders import TextLoader # Load văn bản để LangChain có thể làm việc
from langchain_text_splitters import CharacterTextSplitter # Chia nhỏ thành phần để Huấn luyện
from langchain_chroma import Chroma

In [ ]:
import pandas as pd

df = pd.read_csv('books_cleaned.csv')
df.head()

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."


In [ ]:
df['tagged_description']

,tagged_description
0,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982 A new 'Christie for Christmas' -...
2,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897 Lewis' work on the nature of lov...
4,"9780006280934 ""In The Problem of Pain, C.S. Le..."
...,...
5192,9788172235222 On A Train Journey Home To North...
5193,9788173031014 This book tells the tale of a ma...
5194,9788179921623 Wisdom to Create a Life of Passi...
5195,9788185300535 This collection of the timeless ...


In [ ]:
df['tagged_description'].to_csv('tagged_description.txt',
                                sep = "\n",
                                index = False,
                                header = False)

In [ ]:
raw_doc = TextLoader('tagged_description.txt').load()
text_splitter = CharacterTextSplitter(chunk_size = 1,
                                      chunk_overlap = 0,
                                      separator="\n")

doc = text_splitter.split_documents(raw_doc)

Streaming output truncated to the last 5000 lines.


# 🧠 **Xây dựng Hệ thống**

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# tạo vector DB
db_books = Chroma.from_documents(doc, embedding=embedding)

/tmp/ipykernel_2384/648914237.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [44]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"batch_size": 32}
)

query = "A book to teach children about nature"

results = db_books.similarity_search(query, k=5)

for r in results:
    print(r.page_content)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


9780786808069 Children will discover the exciting world of their own backyard in this introduction to familiar animals from cats and dogs to bugs and frogs. The combination of photographs, illustrations, and fun facts make this an accessible and delightful learning experience.
9780786808380 Introduce your babies to birds, cats, dogs, and babies through fine art, illustration, and photographs. These books are a rare opportunity to expose little ones to a range of images on a single subject, from simple child's drawings and abstract art to playful photos. A brief text accompanies each image, introducing the baby to some basic -- and sometimes playful -- information about the subjects.
9780786808397 Introduce your baby to birds, cats, dogs, and babies through fine art, illustration, and photographs. These books are a rare opportunity to exopse little ones to a range of images on a single subject, from simple child's drawings and abstract art to playful photos. A brief text accompanies eac

In [48]:
df[df['isbn13'] == int(results[0].page_content.split()[0].strip())] # Kết quả

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
3747,9780786808069,0786808063,Baby Einstein: Neighborhood Animals,Marilyn Singer;Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=X9a4P...,Children will discover the exciting world of t...,2001.0,3.89,16.0,180.0,Baby Einstein: Neighborhood Animals,9780786808069 Children will discover the excit...


In [64]:
def retrive_sematic_recommendations(query: str, top_k: int) -> pd.DataFrame: # Dựng hàm recommendation
    results = db_books.similarity_search(query, k=50)

    books_list = []
    for i in range(len(results)):
        raw = results[i].page_content.strip().split()[0]
        clean = raw.strip('"')
        books_list.append(int(clean))

    return df[df['isbn13'].isin(books_list)].head(top_k)

In [65]:
retrive_sematic_recommendations("A book to teach children about nature", 5)

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
59,9780007151240,0007151241,The Family Way,Tony Parsons,Parenthood,http://books.google.com/books/content?id=dJEIx...,It should be the most natural thing in the wor...,2005.0,3.51,400.0,2095.0,The Family Way,9780007151240 It should be the most natural th...
324,9780060959036,0060959037,Prodigal Summer,Barbara Kingsolver,Fiction,http://books.google.com/books/content?id=06IwG...,Barbara Kingsolver's fifth novel is a hymn to ...,2001.0,4.00,444.0,85440.0,Prodigal Summer: A Novel,9780060959036 Barbara Kingsolver's fifth novel...
334,9780060976118,006097611X,Operation Wandering Soul,Richard Powers,Fiction,http://books.google.com/books/content?id=nIGIm...,"Highly imaginative and emotionally powerful, t...",1994.0,3.62,352.0,366.0,Operation Wandering Soul,9780060976118 Highly imaginative and emotional...
383,9780061144899,0061144894,When the Heart Waits,Sue Monk Kidd,Religion,http://books.google.com/books/content?id=JlP91...,From the Bestselling Author of The Secret Life...,2006.0,4.17,240.0,2141.0,When the Heart Waits: Spiritual Direction for ...,9780061144899 From the Bestselling Author of T...
404,9780064402453,0064402452,Racso and the Rats of NIMH,Jane Leslie Conly,Juvenile Fiction,http://books.google.com/books/content?id=MgoNv...,"‘Racso, a brash and boastful little rodent, is...",1988.0,3.76,288.0,3231.0,Racso and the Rats of NIMH,"9780064402453 ‘Racso, a brash and boastful lit..."
